# Optional extension — Mechanical waves and sound

## Mekanik dalgalar ve ses

**Role in the course:** Beyond the scheduled calendar. Travelling waves, superposition, standing waves and sound.

This notebook is **not** a scheduled calendar week. Use it for deeper reading, extra examples and practice after the related weekly notebook. Its problems keep the identifier *Module 13 Pn* for the solution collection.

**TR:** Bu not takvimde ayrı bir hafta değildir; ilgili haftadan sonra ek okuma ve alıştırma için kullanılır.

## Contents / İçindekiler

1. [Before you start](#x13-before)
   · [Setup for the interactive graphs (run once)](#x13-setup)
2. [Concepts, demonstrations and worked examples](#x13-concepts) — 0 worked examples, 5 interactive graphs
3. [Problem set with step-by-step answers](#x13-problems) — 10 problems

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)

<a id="x13-before"></a>

## 1. Before you start / Başlamadan önce

### 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Describe** the mathematical form of a travelling wave and identify its parameters (amplitude, wavelength, frequency, wave speed)
2. **Derive** the wave speed on a string from tension and linear mass density
3. **Explain** superposition, constructive and destructive interference
4. **Visualise** two-source interference patterns in 2-D
5. **Analyse** standing waves and resonance on a string
6. **Model** sound as a longitudinal pressure wave and relate intensity to decibels
7. **Apply** wave concepts to vibration and acoustics problems relevant to engineering

### Joining this lesson / Derse buradan başlayanlar

**Quick recap.** Frequency $f$ counts cycles per second, period $T=1/f$ is seconds per cycle, and wavelength $\lambda$ is the distance between matching points of a wave. Use $v=\lambda f$. For an ideal tensioned string, $v=\sqrt{F_T/\mu}$ and fixed ends give $f_n=nv/(2L)$. In-phase coherent sources interfere constructively when their path difference is an integer wavelength; complete cancellation also needs equal arriving amplitudes. A moving wave pattern is not the same as a moving material particle.

**One measurement, shared engineering questions.** An ultrasonic sensor receives an echo 0.020 s after emission. Assuming a stationary target and constant sound speed 343 m/s, the sound makes a return trip: $2d=vt$, so $d=343(0.020)/2=3.43\,\mathrm{m}$. A mechanical/mechatronics engineer checks the target geometry and possible extra reflections. A software/computer engineer interpreting timestamps checks milliseconds versus seconds and the factor of two. Both should question an apparent 6.86 m distance before trusting a display.

**TR:** Ses hedefe gidip geri gelir; süre iki yolun toplamıdır. Bir ölçümün anlamını açıklamak, ekrandaki sayıyı olduğu gibi kabul etmekten önce gelir.

---

### Before we calculate: a travelling pattern / Dalga fikrine geçiş

**Core route:** label a wave → relate speed, frequency and wavelength → compare path differences → apply endpoint conditions. Fourier decomposition and coding are optional extensions. The graph is a measuring instrument; reading it is the learning task.

Mark two adjacent crests on a *position* graph to measure wavelength $\lambda$ (m). Mark two successive crests at one point on a *time* graph to measure period $T$ (s). Do not read a time from a distance axis. A marked string particle moves up/down; the pattern can travel right/left.

**Algebra bridge:** $v=\lambda f$ gives $\lambda=v/f$ or $f=v/\lambda$. For tension $T_{\rm tension}$, $v=\sqrt{T_{\rm tension}/\mu}$ gives $T_{\rm tension}=\mu v^2$. The same letter $T$ also denotes a period elsewhere; units distinguish them. Convert g→kg and mm→m *before* calculating $\mu=m/L$ or an area.

**Interference is a ratio test:** calculate $q=\Delta r/\lambda$. With in-phase sources, integer $q$ means constructive interference, half-integer $q$ means destructive interference, and other values mean partial interference. Complete cancellation also needs equal arriving amplitudes. “Coherent” means a fixed phase difference; it need not be zero.

**Five-minute pause:** Double the frequency on a string whose tension and density stay fixed. Wave speed stays fixed, so wavelength halves. The demonstration that lets you set frequency and wavelength independently describes a mathematical family; a fixed physical string constrains their product.

**TR:** Dalga boyu uzayda, periyot zamanda ölçülür. Önce neyin dalgalandığını ve dalganın hangi ortamda ilerlediğini söyle; sonra sayıları formüle yerleştir.

<a id="x13-setup"></a>

## Setup for the interactive graphs (run once) / Kurulum — bir kez çalıştır

Run the cells in this section once per session, then run any **Run the demonstration** cell below. Each demonstration shows a status line under its controls: **Updating…** while the graph is drawn, then the draw time and whether the sliders update live or on release. Nothing here needs to be edited.  
**TR:** Bu bölümdeki hücreleri oturum başına bir kez çalıştır; sonra istediğin gösterimi çalıştır. Kontrollerin altındaki durum satırı, grafiğin ne zaman güncellendiğini gösterir.

In [ ]:
#@title Run once — prepare the physics demonstrations
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import cm
from IPython.display import HTML, display, Markdown
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox

%matplotlib inline

# For Colab compatibility
try:
    import google.colab
    IN_COLAB = True
    from matplotlib import rc
    rc('animation', html='jshtml')
except ImportError:
    IN_COLAB = False

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 5)})
print("\u2705 All libraries loaded successfully!")

In [ ]:
#@title Run once — prepare the demonstration controls
"""Shared demonstration interface, embedded in every PHY101 notebook.

Only standard ipywidgets, IPython and matplotlib are used, so the notebooks stay
self-contained in Colab and in a local Jupyter. The design goals are:

* A slider change must always produce a visible reaction. The status line under
  the controls says "Updating…" immediately and reports the draw time afterwards.
* The graph is replaced through the same Output-widget route that
  ``ipywidgets.interact`` uses (``clear_output(wait=True)`` followed by a fresh
  display), which is the most widely tested path in Colab and Jupyter. No
  output-capturing context is used: ipykernel 7 dispatches widget messages
  concurrently and IPython's capture object breaks that dispatch.
* Live updates while dragging are switched on when a graph draws quickly and
  switched off (update on release) when it draws slowly, so the kernel never
  falls behind a fast slider.
"""
import functools
import sys
import time
import traceback

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import HTML, clear_output, display

# Force the inline backend. Otherwise a local kernel may choose a desktop
# backend and block at plt.show(), which looks like a frozen notebook.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

_physics_panels = []
_physics_callback_errors = []

# Draw-time thresholds (seconds) for switching live dragging on and off.
PHYSICS_LIVE_ON = 0.12
PHYSICS_LIVE_OFF = 0.25


def physics_frames(frame_count, maximum=60):
    """Sample display frames, retaining both endpoints and all simulation data."""
    count = int(frame_count)
    shown = min(count, maximum)
    if shown <= 1:
        return list(range(shown))
    return [round(index * (count - 1) / (shown - 1)) for index in range(shown)]


def physics_interval(frame_count, interval_ms):
    """Preserve first-to-last playback duration when display frames are sampled."""
    shown = len(physics_frames(frame_count))
    return interval_ms if shown <= 1 else interval_ms * (int(frame_count) - 1) / (shown - 1)


PHYSICS_STYLE = """<style>
.phy101-panel { border: 1px solid #a9b9c9; border-radius: 8px; padding: 8px; background: #fff; }
.phy101-controls { padding: 0 0 2px; box-sizing: border-box; }
.phy101-status { font-size: 12px; color: #4a5a6a; padding: 0 2px 6px; min-height: 18px; }
.phy101-status.busy { color: #b45309; }
.phy101-plot-output img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-plot-output .output_area { overflow: visible; }
.phy101-plot-output table { font-size: 13px; width: 100%; }
.phy101-plot-output .animation { max-width: 100%; }
.phy101-plot-output .animation img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-animation-panel { max-width: 100%; }
.phy101-animation-panel .animation { display: flex; flex-direction: column; }
.phy101-animation-panel .animation img { order: 2; max-width: 100%; height: auto; }
.phy101-animation-panel .anim-controls { order: 1; background: white; color: #172433; padding: 4px; }
@media (max-width: 650px) {
  .phy101-controls, .phy101-plot-output { width: 100% !important; }
}
</style>"""
display(HTML(PHYSICS_STYLE))


def physics_animation_html(animation):
    """A self-contained animation pane with playback controls kept in view."""
    return HTML(PHYSICS_STYLE + '<div class="phy101-animation-panel">' +
                animation.to_jshtml(default_mode="once") + "</div>")


def _physics_controls(items):
    """Lay the controls out as a wrapping toolbar with full-length labels."""
    flat = []
    for item in items:
        if isinstance(item, (widgets.HBox, widgets.VBox)):
            flat.extend(item.children)
        else:
            flat.append(item)
    for control in flat:
        if hasattr(control, "style") and "description_width" in control.style.traits():
            control.style.description_width = "initial"
        control.layout.width = "310px"
        control.layout.flex = "0 1 310px"
        control.layout.max_width = "100%"
        control.layout.min_width = "0"
        control.layout.margin = "2px 6px 2px 0"
        if isinstance(control, widgets.Button):
            control.layout.width = "auto"
            control.layout.flex = "0 0 auto"
    # Repeat the style inside the widget tree: Colab isolates output frames.
    style = widgets.HTML(value=PHYSICS_STYLE, layout=widgets.Layout(display="none"))
    box = widgets.Box([style] + flat, layout=widgets.Layout(
        display="flex", flex_flow="row wrap", align_items="center",
        width="100%", min_width="0", max_width="100%"))
    box.add_class("phy101-controls")
    return box


def _physics_output(output):
    output.layout = widgets.Layout(
        width="100%", min_width="0", max_width="100%",
        height="auto", overflow="visible", margin="0")
    output.add_class("phy101-plot-output")
    return output


def physics_panel(controls, output):
    """Button-driven demos: a compact control toolbar directly above the result."""
    panel = widgets.Box([_physics_controls(controls), _physics_output(output)],
        layout=widgets.Layout(display="flex", flex_flow="column",
                              align_items="stretch", width="100%"))
    panel.add_class("phy101-panel")
    return panel


def physics_show_figure(figure):
    """Display one inline figure and close its pyplot registration afterwards."""
    import matplotlib.pyplot as plt
    display(figure)
    plt.close(figure)


def physics_vector_axes(axes, points):
    """Equal x/y scales and limits covering all arrow endpoints, including sums."""
    import numpy as np
    coordinates = np.asarray(points, dtype=float).reshape(-1, 2)
    span = max(1.0, float(np.max(np.abs(coordinates)))) * 1.22
    axes.set(xlim=(-span, span), ylim=(-span, span), xlabel="x component", ylabel="y component")
    axes.set_aspect("equal", adjustable="box")
    axes.axhline(0, color="#718096", linewidth=0.7)
    axes.axvline(0, color="#718096", linewidth=0.7)
    axes.grid(alpha=0.2)


class PhysicsPanel:
    """Controls, a status line and one Output widget that shows the latest result."""

    def __init__(self, function, controls):
        self.f = function
        self.controls = controls
        self.out = _physics_output(widgets.Output())
        self.status = widgets.HTML(value="")
        self.status.add_class("phy101-status")
        self.seconds = None
        self.live = True
        self.updates = 0
        self.last_outputs = 0
        self.figures = 0
        self.error = None
        visible = []
        for control in controls.values():
            if isinstance(control, widgets.fixed):
                continue
            visible.append(control)
            if hasattr(control, "continuous_update"):
                control.continuous_update = True
            control.observe(self._changed, names="value")
        self.widget = widgets.VBox([_physics_controls(visible), self.status, self.out],
                                   layout=widgets.Layout(width="100%"))
        self.widget.add_class("phy101-panel")
        self.children = self.widget.children
        _physics_panels.append(self)
        self.render()

    # Compatibility with the earlier validation code.
    @property
    def layout(self):
        return self.widget.layout

    def _changed(self, change):
        self.render()

    def _set_live(self, live):
        if live == self.live:
            return
        self.live = live
        for control in self.controls.values():
            if hasattr(control, "continuous_update"):
                control.continuous_update = live

    def render(self):
        self.status.value = "⏳ Updating… / Güncelleniyor…"
        self.status.add_class("busy")
        started = time.perf_counter()
        kwargs = {name: control.value for name, control in self.controls.items()}
        self.error = None
        self.figures = 0
        # Count everything the demonstration shows (figures, HTML, animations, text)
        # by wrapping the display publisher and stdout for this draw only.
        shell = get_ipython()
        publisher = getattr(shell, "display_pub", None) if shell is not None else None
        if publisher is not None:
            original_publish = publisher.publish

            def counting_publish(*args, **kwargs):
                self.figures += 1
                return original_publish(*args, **kwargs)
            publisher.publish = counting_publish
        stdout = sys.stdout
        original_write = stdout.write

        def counting_write(text):
            if text.strip():
                self.figures += 1
            return original_write(text)
        stdout.write = counting_write
        try:
            # The previous result stays visible until the new one arrives.
            with self.out:
                clear_output(wait=True)
                try:
                    result = self.f(**kwargs)
                    from ipywidgets.widgets.interaction import show_inline_matplotlib_plots
                    show_inline_matplotlib_plots()
                    if result is not None:
                        display(result)
                except Exception:
                    self.error = traceback.format_exc()
                    _physics_callback_errors.append((getattr(self.f, "__name__", "callback"), self.error))
                    print(self.error)
        finally:
            if publisher is not None and publisher.__dict__.get("publish") is counting_publish:
                del publisher.publish
            if stdout.__dict__.get("write") is counting_write:
                del stdout.write
        self.last_outputs = self.figures
        self.seconds = time.perf_counter() - started
        self.updates += 1
        if self.seconds > PHYSICS_LIVE_OFF:
            self._set_live(False)
        elif self.seconds < PHYSICS_LIVE_ON:
            self._set_live(True)
        self.status.remove_class("busy")
        mode = ("updates while you drag / sürüklerken güncellenir" if self.live
                else "updates when you release the slider / kaydırıcıyı bırakınca güncellenir")
        self.status.value = (f"✓ Drawn in {self.seconds:.2f} s · {mode}" if self.error is None
                             else "⚠ The demonstration reported an error; see the message below.")


def physics_interactive(function, **controls):
    """Build a panel like ipywidgets.interactive, returning the panel object."""
    @functools.wraps(function)
    def checked(*args, **kwargs):
        return function(*args, **kwargs)

    return PhysicsPanel(checked, controls)


def physics_interact(function=None, **controls):
    """Support both @physics_interact(...) and physics_interact(function, ...)."""
    if function is None:
        return lambda function: physics_interact(function, **controls)
    panel = physics_interactive(function, **controls)
    function.widget = panel.widget
    function.panel = panel
    display(panel.widget)
    return function


<a id="x13-concepts"></a>

## 2. Concepts, demonstrations and worked examples / Konular, gösterimler ve çözümlü örnekler

### What is a Wave?

A **wave** is a disturbance that transfers **energy** without transferring **matter**.

<table width="100%">
<thead>
<tr>
<th width="160" scope="col">Property</th>
<th width="88" scope="col">Symbol</th>
<th width="64" scope="col">Unit</th>
<th width="416" scope="col">Meaning</th>
</tr>
</thead>
<tbody>
<tr>
<td>Amplitude</td>
<td>$A$</td>
<td>$\mathrm{m}$</td>
<td>Maximum displacement from equilibrium</td>
</tr>
<tr>
<td>Wavelength</td>
<td>$\lambda$</td>
<td>$\mathrm{m}$</td>
<td>Distance between two consecutive identical points</td>
</tr>
<tr>
<td>Frequency</td>
<td>$f$</td>
<td>$\mathrm{Hz}$</td>
<td>Number of complete cycles per second</td>
</tr>
<tr>
<td>Period</td>
<td>$T = 1/f$</td>
<td>$\mathrm{s}$</td>
<td>Time for one complete cycle</td>
</tr>
<tr>
<td>Wave speed</td>
<td>$v = \lambda f$</td>
<td>$\mathrm{m/s}$</td>
<td>Speed at which the disturbance travels</td>
</tr>
<tr>
<td>Wave number</td>
<td>$k = \frac{2\pi}{\lambda}$</td>
<td>$\mathrm{rad/m}$</td>
<td>Spatial frequency</td>
</tr>
<tr>
<td>Angular frequency</td>
<td>$\omega = 2\pi f$</td>
<td>$\mathrm{rad/s}$</td>
<td>Temporal frequency</td>
</tr>
</tbody>
</table>

#### The Travelling Wave Equation

$$y(x,t) = A \sin(kx - \omega t + \phi)$$

- $kx - \omega t$ : wave travelling in the **+x** direction
- $kx + \omega t$ : wave travelling in the **-x** direction
- $\phi$ : initial phase

#### Wave Speed on a String

$$v = \sqrt{\frac{T}{\mu}}$$

where $T$ is the tension (N) and $\mu$ is the linear mass density (kg/m).

> **Engineering Analogy:** Think of a wave on a string like a ripple of information travelling through a power line. The tighter the cable (higher tension), the faster disturbances travel.

#### A wave equation you can read / Dalga denklemini okumak

For a class example, use
$$y(x,t)=(0.020\,\mathrm m)\sin[(4\pi\,\mathrm{rad/m})x-(8\pi\,\mathrm{rad/s})t].$$
Compare it term by term with $A\sin(kx-\omega t)$:
$$A=0.020\,\mathrm m,\quad k=4\pi\,\mathrm{rad/m},\quad\omega=8\pi\,\mathrm{rad/s}.$$
$$\lambda=\frac{2\pi}{k}=0.50\,\mathrm m,\qquad
f=\frac{\omega}{2\pi}=4.0\,\mathrm{Hz},\qquad
v=\lambda f=2.0\,\mathrm{m/s}.$$
Follow a crest of fixed phase: $kx-\omega t=\text{constant}$ implies $x=(\omega/k)t+\text{constant}$. Its position increases with time, so the pattern travels right. A marked particle on the string oscillates vertically around its fixed horizontal position.

**Türkçe:** Sinüsün önündeki sayı genliktir. $x$’in katsayısı dalga sayısını, $t$’nin katsayısı açısal frekansı verir; bunları doğrudan dalga boyu ve frekans sanma. Dalganın ilerlemesi ile ipin bir noktasının hareketi farklıdır.

### 🎬 Interactive: Animated Transverse Wave

Watch how a sinusoidal wave propagates along a string. Use the sliders to change wavelength ($\lambda$) and frequency ($f$) and observe how the wave speed $v = \lambda f$ changes.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def transverse_wave_animation(wavelength=2.0, frequency=1.0, amplitude=1.0):
    """Animate a transverse wave with given parameters."""
    fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
    x = np.linspace(0, 10, 500)
    k = 2 * np.pi / wavelength
    omega = 2 * np.pi * frequency
    v = wavelength * frequency

    line, = ax.plot([], [], 'b-', lw=2.5)
    dot, = ax.plot([], [], 'ro', ms=10, zorder=5)
    time_text = ax.text(0.98, 0.97, '', transform=ax.transAxes, ha='right', va='top', fontsize=10,
                        bbox=dict(boxstyle='round', facecolor='lightyellow'))

    ax.set_xlim(0, 10)
    ax.set_ylim(-2.3, 2.1)
    ax.set_xlabel('Position x (m)')
    ax.set_ylabel('Displacement y (m)')
    ax.set_title(f'Transverse Wave:  $\\lambda$={wavelength:.1f} m,  f={frequency:.1f} Hz,  v={v:.1f} m/s')
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    ax.grid(True, alpha=0.3)

    # Draw wavelength annotation
    ax.annotate('', xy=(wavelength, -1.85), xytext=(0, -1.85),
                arrowprops=dict(arrowstyle='<->', color='green', lw=2))
    ax.text(wavelength/2, -2.12, f'$\\lambda$ = {wavelength:.1f} m', ha='center',
            color='green', fontsize=11, fontweight='bold')

    n_frames = 80
    T_total = 2.0 / frequency if frequency > 0 else 2.0

    def init():
        line.set_data([], [])
        dot.set_data([], [])
        time_text.set_text('')
        return line, dot, time_text

    def animate(i):
        t = i * T_total / n_frames
        y = amplitude * np.sin(k * x - omega * t)
        line.set_data(x, y)
        # Track a single point on the string
        x_pt = 5.0
        y_pt = amplitude * np.sin(k * x_pt - omega * t)
        dot.set_data([x_pt], [y_pt])
        time_text.set_text(f't = {t:.2f} s')
        return line, dot, time_text

    ani = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=physics_frames(n_frames), interval=physics_interval(n_frames, 40), blit=True)
    plt.close(fig)
    return ani

# Interactive widget version
@physics_interact(wavelength=FloatSlider(min=0.5, max=5.0, step=0.25, value=2.0, description='$\\lambda$ (m)'),
          frequency=FloatSlider(min=0.25, max=3.0, step=0.25, value=1.0, description='f (Hz)'),
          amplitude=FloatSlider(min=0.2, max=1.5, step=0.1, value=1.0, description='A (m)'))
def show_wave(wavelength, frequency, amplitude):
    ani = transverse_wave_animation(wavelength, frequency, amplitude)
    display(physics_animation_html(ani))

### Superposition Principle

When two or more waves overlap in the same medium, the **net displacement** is the **algebraic sum** of the individual displacements:

$$y_{\text{net}}(x,t) = y_1(x,t) + y_2(x,t)$$

<table width="100%">
<thead>
<tr>
<th width="224" scope="col">Type</th>
<th width="296" scope="col">Condition</th>
<th width="200" scope="col">Result</th>
</tr>
</thead>
<tbody>
<tr>
<td><strong>Constructive</strong> interference</td>
<td>Waves in phase ($\Delta\phi = 0, 2\pi, \dots$)</td>
<td>Amplitude doubles</td>
</tr>
<tr>
<td><strong>Destructive</strong> interference</td>
<td>Waves out of phase ($\Delta\phi = \pi, 3\pi, \dots$)</td>
<td>Amplitude cancels</td>
</tr>
<tr>
<td><strong>Partial</strong> interference</td>
<td>Other phase differences</td>
<td>Intermediate amplitude</td>
</tr>
</tbody>
</table>

The “doubles” and “cancels” statements in this table assume **equal-amplitude, equal-frequency** waves at the observation point. With unequal amplitudes the minimum is $|A_1-A_2|$, not zero. **TR:** Tam sönüm için zıt fazın yanında eşit genlik de gerekir.

#### Think–Pair–Explain: add signed displacements

At one place and time, two small waves separately give $y_1=+3.0\,\mathrm{mm}$ and $y_2=-1.0\,\mathrm{mm}$. Predict the combined displacement, then compare with a partner.
$$y_{\rm net}=y_1+y_2=(+3.0)+(-1.0)=+2.0\,\mathrm{mm}.$$
The result remains above equilibrium; it is smaller because the displacements oppose. If the second contribution were $-3.0\,\mathrm{mm}$, the instantaneous displacement would be zero. That alone does not prove energy has disappeared or that cancellation persists at other times.

**Türkçe:** Genlikleri işaretsiz toplayarak başlamıyoruz; aynı yer ve zamandaki yer değiştirmeleri işaretleriyle topluyoruz. Sıfır toplam yer değiştirme, ortamın her yerde durduğu ya da enerjinin yok olduğu anlamına gelmez.

**Same-medium comparison.** In the demonstration both travelling waves have $v=2\,\mathrm{m/s}$. Changing $f_2$ changes $\lambda_2=v/f_2$ as well as its time variation. Türkçe: aynı dağılmasız ortamda frekansı değiştirip dalga boyunu sabit bırakamayız; hızın aynı kalması için dalga boyu ters orantılı değişir.

### 🎬 Interactive: Superposition of Two Waves

See how two waves combine. Adjust the frequency and phase of the second wave to observe constructive and destructive interference.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def superposition_animation(f2=1.0, phase_shift=0.0):
    """Animate the superposition of two waves."""
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), layout="constrained", sharex=True)
    x = np.linspace(0, 10, 500)
    f1 = 1.0
    lam = 2.0
    k = 2 * np.pi / lam
    omega1 = 2 * np.pi * f1
    omega2 = 2 * np.pi * f2
    wave_speed = lam * f1
    k2 = omega2 / wave_speed  # same nondispersive medium, so wavelength changes with frequency

    line1, = axes[0].plot([], [], 'b-', lw=2, label='Wave 1')
    line2, = axes[1].plot([], [], 'r-', lw=2, label='Wave 2')
    line3, = axes[2].plot([], [], 'purple', lw=2.5, label='Superposition')

    for i, (ax, title) in enumerate(zip(axes, ['Wave 1 (reference)', f'Wave 2 (f={f2:.1f} Hz, $\\Delta\\phi$={phase_shift:.1f} rad)', 'Superposition = Wave 1 + Wave 2'])):
        ax.set_xlim(0, 10)
        ax.set_ylim(-2.5, 2.5)
        ax.axhline(0, color='gray', ls='--', lw=0.5)
        ax.set_title(title, fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_ylabel('y (m)')
    axes[2].set_xlabel('Position x (m)')

    n_frames = 80
    T_total = 2.0

    def animate(i):
        t = i * T_total / n_frames
        y1 = np.sin(k * x - omega1 * t)
        y2 = np.sin(k2 * x - omega2 * t + phase_shift)
        line1.set_data(x, y1)
        line2.set_data(x, y2)
        line3.set_data(x, y1 + y2)
        return line1, line2, line3

    ani = animation.FuncAnimation(fig, animate, frames=physics_frames(n_frames), interval=physics_interval(n_frames, 40), blit=True)
    plt.close(fig)
    return ani

@physics_interact(f2=FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description='f$_2$ (Hz)'),
          phase_shift=FloatSlider(min=0, max=2*np.pi, step=0.1, value=0, description='$\\Delta\\phi$ (rad)'))
def show_superposition(f2, phase_shift):
    ani = superposition_animation(f2, phase_shift)
    display(physics_animation_html(ani))

#### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** Two waves give $+3.0\,\mathrm{mm}$ and $-1.0\,\mathrm{mm}$ at the same position and time.

**Think — 1 minute:** Predict the sign and size of their sum before using a calculator.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

Superposition adds signed displacements:
$$y=y_1+y_2=3.0+(-1.0)=+2.0\,\mathrm{mm}.$$
The contributions oppose but do not cancel completely. Their individual maximum amplitudes cannot replace these instantaneous signed displacements.

**Türkçe:** Aynı yer ve zamana ait değerleri topladık. İkinci dalga aşağı yönde olduğu için eksi işaretlidir. Sonuç yukarı yönde kalır; bu yüzden “iki dalga var, genlik iki kat” demek burada yanlıştır.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### Two-Source Interference (2-D)

When two point sources emit waves of the same frequency, they create an **interference pattern** in 2-D space.

At any point $P$, the path difference is:
$$\Delta r = r_2 - r_1$$

<table width="100%">
<thead>
<tr>
<th width="120" scope="col">Condition</th>
<th width="256" scope="col">Path difference</th>
<th width="160" scope="col">Result</th>
</tr>
</thead>
<tbody>
<tr>
<td>Constructive</td>
<td>$\Delta r = m\lambda$ ($m = 0, \pm 1, \pm 2, \dots$)</td>
<td>Maximum intensity</td>
</tr>
<tr>
<td>Destructive</td>
<td>$\Delta r = (m + \tfrac{1}{2})\lambda$</td>
<td>Zero intensity</td>
</tr>
</tbody>
</table>

The resulting pattern shows bright (constructive) and dark (destructive) **fringes** — similar to what you see when dropping two pebbles in a pond.

These path conditions assume sources are **in phase**. A zero-intensity minimum also assumes equal arriving amplitudes. With an initial relative phase $\phi_0$, include it in $\Delta\phi=2\pi\Delta r/\lambda+\phi_0$.

### 🎬 Interactive: Two-Source Interference Pattern (2-D Heatmap)

Visualise the interference pattern from two coherent point sources. Adjust the source **separation** ($d$) and **wavelength** ($\lambda$) to see how the pattern changes.

**Reading the colour map:** the instantaneous field uses a symmetric colour range of $-2$ to $+2$ arbitrary units to keep distant fringes visible. The colour-bar arrows mark saturation near a source; they do not indicate a physical maximum. The phase control is an angle in radians. Türkçe: kırmızı veya mavinin doygunluğu, kaynağın yakınındaki genliği sayısal olarak okumaya yetmez; desenin faz ilişkisini gösterir.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def two_source_interference(separation=3.0, wavelength=1.0, t_phase=0.0):
    """Plot 2D interference pattern from two point sources."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")

    # Grid
    N = 400
    x = np.linspace(-8, 8, N)
    y = np.linspace(-8, 8, N)
    X, Y = np.meshgrid(x, y)

    # Source positions on the y-axis
    s1 = np.array([0, -separation / 2])
    s2 = np.array([0,  separation / 2])

    k = 2 * np.pi / wavelength
    # t_phase is phase in radians, as labelled by the control.

    r1 = np.sqrt((X - s1[0])**2 + (Y - s1[1])**2)
    r2 = np.sqrt((X - s2[0])**2 + (Y - s2[1])**2)

    # Avoid division by zero at sources
    r1 = np.maximum(r1, 0.05)
    r2 = np.maximum(r2, 0.05)

    # Wave fields (with 1/sqrt(r) decay for 2D circular waves)
    psi1 = np.sin(k * r1 - t_phase) / np.sqrt(r1)
    psi2 = np.sin(k * r2 - t_phase) / np.sqrt(r2)
    psi_total = psi1 + psi2

    # Left: instantaneous wave field
    im1 = axes[0].imshow(psi_total, extent=[-8, 8, -8, 8], cmap='RdBu_r',
                          vmin=-2, vmax=2, origin='lower')
    axes[0].plot(*s1, 'ko', ms=8, label='Source 1')
    axes[0].plot(*s2, 'ko', ms=8, label='Source 2')
    axes[0].set_title('Instantaneous Wave Field', fontsize=12)
    axes[0].set_xlabel('x (m)')
    axes[0].set_ylabel('y (m)')
    axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
    plt.colorbar(im1, ax=axes[0], label='Scaled displacement (arb. units)', extend='both', shrink=0.85, pad=0.03)

    # Right: time-averaged intensity
    # I ~ |A1/sqrt(r1) + A2/sqrt(r2) * e^{i*k*delta_r}|^2
    intensity = 1/r1 + 1/r2 + 2*np.cos(k*(r2 - r1)) / np.sqrt(r1 * r2)
    im2 = axes[1].imshow(intensity, extent=[-8, 8, -8, 8], cmap='hot',
                          origin='lower', vmin=0)
    axes[1].plot(*s1, 'wo', ms=8)
    axes[1].plot(*s2, 'wo', ms=8)
    axes[1].set_title('Time-Averaged Intensity', fontsize=12)
    axes[1].set_xlabel('x (m)')
    axes[1].set_ylabel('y (m)')
    plt.colorbar(im2, ax=axes[1], label='Intensity (arb. units)', shrink=0.85, pad=0.03)

    fig.suptitle(f'd = {separation:.1f} m,   $\\lambda$ = {wavelength:.2f} m,   d/$\\lambda$ = {separation/wavelength:.1f}',
                 fontsize=13, fontweight='bold')
    plt.show()

@physics_interact(separation=FloatSlider(min=0.5, max=6.0, step=0.25, value=3.0, description='d (m)'),
          wavelength=FloatSlider(min=0.3, max=3.0, step=0.1, value=1.0, description='$\\lambda$ (m)'),
          t_phase=FloatSlider(min=0, max=2*np.pi, step=0.2, value=0, description='Phase (rad)'))
def show_interference(separation, wavelength, t_phase):
    two_source_interference(separation, wavelength, t_phase)

### Standing Waves

When a wave reflects off a boundary, the incident and reflected waves can **superpose** to form a **standing wave**:

$$y(x,t) = 2A \sin(kx) \cos(\omega t)$$

Key features:
- **Nodes**: points that never move ($\sin(kx) = 0$)
- **Antinodes**: points of maximum vibration ($\sin(kx) = \pm 1$)

#### Resonant Frequencies on a String (fixed at both ends)

$$f_n = \frac{n}{2L} \sqrt{\frac{T}{\mu}}, \quad n = 1, 2, 3, \dots$$

<table width="100%">
<thead>
<tr>
<th width="88" scope="col">Harmonic</th>
<th width="64" scope="col">$n$</th>
<th width="104" scope="col">Wavelength</th>
<th width="152" scope="col">Name</th>
</tr>
</thead>
<tbody>
<tr>
<td>1st</td>
<td>1</td>
<td>$\lambda_1 = 2L$</td>
<td>Fundamental</td>
</tr>
<tr>
<td>2nd</td>
<td>2</td>
<td>$\lambda_2 = L$</td>
<td>1st overtone</td>
</tr>
<tr>
<td>3rd</td>
<td>3</td>
<td>$\lambda_3 = \frac{2L}{3}$</td>
<td>2nd overtone</td>
</tr>
<tr>
<td>$n$th</td>
<td>$n$</td>
<td>$\lambda_n = \frac{2L}{n}$</td>
<td>$(n-1)$th overtone</td>
</tr>
</tbody>
</table>

> **Engineering Analogy:** Standing waves on a guitar string produce musical notes. The fundamental frequency determines the pitch, while higher harmonics give the instrument its characteristic tone (timbre). The same principle applies to vibration analysis of bridges and buildings!

#### Fit half-wavelengths between the ends / Uç koşulunu çiz

For a string of length $L=1.20\,\mathrm m$ and wave speed $v=48\,\mathrm{m/s}$, both ends must be nodes. In mode $n=3$, three half-wavelengths fill the length:
$$L=3\frac{\lambda_3}{2}\quad\Rightarrow\quad
\lambda_3=\frac{2L}{3}=0.80\,\mathrm m,\qquad
f_3=\frac{v}{\lambda_3}=\frac{48}{0.80}=60\,\mathrm{Hz}.$$
There are four nodes including the two ends and three antinodes. Nodes are at $x=0,\ 0.40,\ 0.80,\ 1.20\,\mathrm m$; adjacent nodes are separated by $\lambda_3/2$, not a full wavelength.

**Türkçe:** Önce iki ucun hareket edemediğini göster. Üçüncü modda ip üzerine üç yarım dalga sığar. Düğüm sayısını hesaplarken uçları da say; antinodlar en büyük genlikle titreşen konumlardır.

### 🎬 Interactive: Standing Wave Animation

Observe standing waves on a string fixed at both ends. Change the **harmonic number** $n$ to see different modes.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def standing_wave_animation(n_harmonic=1):
    """Animate a standing wave for the n-th harmonic."""
    L = 1.0  # string length
    A = 1.0
    fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")

    x = np.linspace(0, L, 500)
    k = n_harmonic * np.pi / L
    omega = 2 * np.pi * n_harmonic  # proportional to harmonic
    lam = 2 * L / n_harmonic

    # Envelope
    envelope = 2 * A * np.abs(np.sin(k * x))
    ax.fill_between(x, -envelope, envelope, alpha=0.1, color='blue')
    ax.plot(x, envelope, 'b--', lw=1, alpha=0.5)
    ax.plot(x, -envelope, 'b--', lw=1, alpha=0.5)

    line, = ax.plot([], [], 'b-', lw=3)

    # Mark nodes
    nodes_x = np.linspace(0, L, n_harmonic + 1)
    ax.plot(nodes_x, np.zeros_like(nodes_x), 'ko', ms=10, zorder=5, label='Nodes')

    # Mark antinodes
    antinodes_x = (nodes_x[:-1] + nodes_x[1:]) / 2
    ax.plot(antinodes_x, np.zeros_like(antinodes_x), 'r^', ms=10, zorder=5, label='Antinodes')

    ax.set_xlim(-0.04 * L, 1.04 * L)
    ax.set_ylim(-2.5, 2.5)
    ax.set_xlabel('Position along string (m)')
    ax.set_ylabel('Displacement (m)')
    ax.set_title(f'Standing Wave: Harmonic n = {n_harmonic},  $\\lambda$ = {lam:.3f} m\n{n_harmonic} half-wavelength(s) between fixed ends')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
    ax.grid(True, alpha=0.3)

    n_frames = 60

    def animate(i):
        t = i * 2 * np.pi / (omega * n_frames)
        y = 2 * A * np.sin(k * x) * np.cos(omega * t)
        line.set_data(x, y)
        return (line,)

    ani = animation.FuncAnimation(fig, animate, frames=physics_frames(n_frames), interval=physics_interval(n_frames, 40), blit=True)
    plt.close(fig)
    return ani

@physics_interact(n_harmonic=IntSlider(min=1, max=8, step=1, value=1, description='Harmonic n'))
def show_standing(n_harmonic):
    ani = standing_wave_animation(n_harmonic)
    display(physics_animation_html(ani))

### Sound Waves

Sound is a **longitudinal** wave: particles oscillate **parallel** to the direction of propagation, creating alternating regions of **compression** (high pressure) and **rarefaction** (low pressure).

#### Speed of Sound

<table width="100%">
<thead>
<tr>
<th width="128" scope="col">Medium</th>
<th width="112" scope="col">Speed (m/s)</th>
<th width="176" scope="col">Notes</th>
</tr>
</thead>
<tbody>
<tr>
<td>Air (20 $^\circ$C)</td>
<td>343</td>
<td>$v \approx 331 + 0.6\,T_{\text{C}}$</td>
</tr>
<tr>
<td>Water (25 $^\circ$C)</td>
<td>1480</td>
<td>Important for sonar</td>
</tr>
<tr>
<td>Steel</td>
<td>5960</td>
<td>Fastest in solids</td>
</tr>
</tbody>
</table>

#### Intensity and Decibels

Sound intensity level in decibels (dB):

$$\beta = 10 \log_{10}\!\left(\frac{I}{I_0}\right)$$

where $I_0 = 10^{-12}\,\mathrm{W/m^2}$ is the threshold of hearing.

<table width="100%">
<thead>
<tr>
<th width="184" scope="col">Sound</th>
<th width="152" scope="col">Intensity ($\mathrm{W/m^2}$)</th>
<th width="104" scope="col">Level (dB)</th>
</tr>
</thead>
<tbody>
<tr>
<td>Threshold of hearing</td>
<td>$10^{-12}$</td>
<td>0</td>
</tr>
<tr>
<td>Whisper</td>
<td>$10^{-10}$</td>
<td>20</td>
</tr>
<tr>
<td>Conversation</td>
<td>$10^{-6}$</td>
<td>60</td>
</tr>
<tr>
<td>Rock concert</td>
<td>$10^{-1}$</td>
<td>110</td>
</tr>
<tr>
<td>Threshold of pain</td>
<td>$1$</td>
<td>120</td>
</tr>
</tbody>
</table>

#### Work through a logarithm / Logaritmayı adım adım aç

For $I=10^{-7}\,\mathrm{W/m^2}$ and reference $I_0=10^{-12}\,\mathrm{W/m^2}$,
$$\frac{I}{I_0}=\frac{10^{-7}}{10^{-12}}=10^{(-7)-(-12)}=10^5,$$
$$\beta=10\log_{10}(10^5)=10(5)=50\,\mathrm{dB}.$$
The intensity units cancel before the logarithm. Increasing intensity tenfold adds $10\,\mathrm{dB}$ because $\log_{10}(10I/I_0)=1+\log_{10}(I/I_0)$; it does not multiply the dB level by ten.

**Türkçe:** Üslü sayıları bölerken üsleri çıkar: $-7-(-12)=5$. Logaritmanın içine birimli bir sayı değil, aynı birimdeki iki yoğunluğun oranı girer. Desibel doğrusal bir enerji ölçeği değildir.

**Schematic display, physical relationship:** the sound animation intentionally slows time and scales the vertical axes independently. It shows phase and back-and-forth particle motion, not measured pascals or metres. Actual sound speed must be found from physical $v=f\lambda$ data, not from this display's speed. Türkçe: tanecikler dalgayla birlikte uçtan uca gitmez; denge noktaları çevresinde salınır. Basınç ile tanecik yer değiştirmesinin ölçekleri farklıdır.

### 🎬 Interactive: Sound Wave Visualisation (Pressure Wave)

Visualise a sound wave as a longitudinal pressure variation. The top panel shows the pressure distribution; the bottom shows particle displacement.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def sound_wave_animation(frequency=2.0, amplitude=1.0):
    """Animate a sound wave showing both pressure and displacement."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), layout="constrained", sharex=True)

    x = np.linspace(0, 10, 500)
    lam = 2.0  # schematic spatial scale; frequency controls the slowed display
    k = 2 * np.pi / lam
    omega = 2 * np.pi * frequency

    # Pressure wave
    line_p, = ax1.plot([], [], 'r-', lw=2.5)
    ax1.set_ylim(-1.8, 1.8)
    ax1.set_ylabel('Normalized pressure', fontsize=11)
    ax1.set_title(f'Sound-wave schematic: display rate {frequency:.1f} cycles/s\nPositive pressure: compression; negative: rarefaction', fontsize=12)
    ax1.axhline(0, color='gray', ls='--', lw=0.5)
    ax1.grid(True, alpha=0.3)

    # Add compression/rarefaction labels

    # Displacement wave (90 deg out of phase with pressure)
    line_d, = ax2.plot([], [], 'b-', lw=2.5)
    ax2.set_ylim(-1.8, 1.8)
    ax2.set_xlabel('Position (schematic units)')
    ax2.set_xlim(-0.4, 10.4)
    ax2.set_ylabel('Normalized particle\ndisplacement', fontsize=11)
    ax2.axhline(0, color='gray', ls='--', lw=0.5)
    ax2.grid(True, alpha=0.3)

    # Particle dots for longitudinal visualization
    n_particles = 50
    x_particles = np.linspace(0.2, 9.8, n_particles)
    dots, = ax2.plot([], [], 'ko', ms=4, alpha=0.7)


    n_frames = 80
    T_total = 2.0 / frequency if frequency > 0 else 2.0

    def animate(i):
        t = i * T_total / n_frames
        # Pressure: p = P_max * sin(kx - wt)
        p = amplitude * np.sin(k * x - omega * t)
        line_p.set_data(x, p)
        # Displacement: s = s_max * cos(kx - wt) -- 90 deg out of phase
        s = amplitude * np.cos(k * x - omega * t)
        line_d.set_data(x, s)
        # Particle positions (displaced)
        s_part = 0.3 * amplitude * np.cos(k * x_particles - omega * t)
        dots.set_data(x_particles + s_part, np.zeros_like(x_particles))
        return line_p, line_d, dots

    ani = animation.FuncAnimation(fig, animate, frames=physics_frames(n_frames), interval=physics_interval(n_frames, 40), blit=True)
    plt.close(fig)
    return ani

@physics_interact(frequency=FloatSlider(min=0.5, max=4.0, step=0.25, value=2.0, description='Display rate'),
          amplitude=FloatSlider(min=0.2, max=1.5, step=0.1, value=1.0, description='A'))
def show_sound(frequency, amplitude):
    ani = sound_wave_animation(frequency, amplitude)
    display(physics_animation_html(ani))

#### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** A fixed-end string has $L=1.20\,\mathrm m$ and $v=48\,\mathrm{m/s}$. It vibrates in its third mode.

**Think — 1 minute:** Sketch the nodes first; decide how many half-wavelengths fit.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

$$L=3\frac{\lambda}{2}
\quad\Longrightarrow\quad\lambda=\frac{2L}{3}=\frac{2(1.20)}{3}=0.80\,\mathrm m,$$
$$f=\frac{v}{\lambda}=\frac{48}{0.80}=60\,\mathrm{Hz}.$$
There are three antinodes and four nodes including the ends. Wave speed remains $48\,\mathrm{m/s}$; changing the mode changes wavelength and frequency together.

**Türkçe:** İpin uçları düğümdür. Mod numarası ip üzerine sığan yarım dalga sayısını verir. Üçüncü moda geçince aynı ortamda dalga hızı üç katına çıkmaz; frekans ve dalga boyu birlikte değişir.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### 📝 Worked Example 1: Wave on a String

**Problem:** A steel guitar string has length $L = 0.65\,\mathrm{m}$, mass $m = 3.5\,\mathrm{g}$, and is under tension $T = 72\,\mathrm{N}$.

Find: (a) the wave speed, (b) the fundamental frequency, (c) the frequency of the 3rd harmonic.

**Step 1 — convert before dividing.**
$$m=3.5\,\mathrm g\times\frac{1\,\mathrm{kg}}{1000\,\mathrm g}=0.0035\,\mathrm{kg},$$
$$\mu=\frac mL=\frac{0.0035\,\mathrm{kg}}{0.65\,\mathrm m}
=0.00538462\,\mathrm{kg/m}.$$

**(a) Take the square root of tension divided by linear density.**

$$v=\sqrt{\frac{T}{\mu}}=\sqrt{\frac{72}{0.00538462}}
=\sqrt{13371.43}=115.635\,\mathrm{m/s}.$$

The units under the root are $\mathrm{m^2/s^2}$, so the result has units of speed.

**(b) Use the fixed-end fundamental wavelength.**

$$\lambda_1=2L=1.30\,\mathrm m,\qquad
f_1=\frac{v}{\lambda_1}=\frac{115.635}{1.30}=88.9499\,\mathrm{Hz}.$$

**(c) Multiply the fundamental frequency by the mode number.**

$$f_3=3f_1=266.850\,\mathrm{Hz}.$$

**Interpretation:** The wave speed is the same for these ideal string modes; shorter wavelengths produce higher frequencies. **TR:** Üçüncü harmonikte hız üç katına çıkmaz; frekans üç katına çıkar, dalga boyu üçte bire iner.

In [ ]:
#@title Optional numerical check — the algebra is explained above
# Worked Example 1: Wave on a guitar string
L = 0.65       # m
m = 3.5e-3     # kg
T = 72         # N

# (a) Linear mass density and wave speed
mu = m / L
v = np.sqrt(T / mu)
print(f"Linear mass density: mu = {mu:.4f} kg/m")
print(f"(a) Wave speed: v = sqrt(T/mu) = sqrt({T}/{mu:.4f}) = {v:.1f} m/s")

# (b) Fundamental frequency
f1 = v / (2 * L)
print(f"\n(b) Fundamental frequency: f1 = v/(2L) = {v:.1f}/(2*{L}) = {f1:.1f} Hz")

# (c) 3rd harmonic
f3 = 3 * f1
print(f"\n(c) 3rd harmonic: f3 = 3*f1 = 3*{f1:.1f} = {f3:.1f} Hz")

### 📝 Worked Example 2: Two-Source Interference

**Problem:** Two speakers separated by $d = 2.0\,\mathrm{m}$ emit sound at $f = 680\,\mathrm{Hz}$ in phase. A listener stands at a point where $r_1 = 4.0\,\mathrm{m}$ and $r_2 = 4.5\,\mathrm{m}$ from the respective speakers. The speed of sound is 340 m/s.

Is the interference constructive, destructive, or partial?

**Step 1 — calculate wavelength.**
$$\lambda=\frac vf=\frac{340\,\mathrm{m/s}}{680\,\mathrm{s^{-1}}}=0.500\,\mathrm m.$$
**Step 2 — subtract the two travelled distances.**
$$\Delta r=r_2-r_1=4.5-4.0=0.500\,\mathrm m.$$
**Step 3 — compare that difference with one wavelength.**
$$q=\frac{\Delta r}{\lambda}=\frac{0.500\,\mathrm m}{0.500\,\mathrm m}=1.$$

One extra wavelength gives an extra full phase cycle, $\Delta\phi=2\pi\Delta r/\lambda=2\pi$. Thus the waves arrive **in phase: constructive interference**. The 2.0 m source separation is not the listener's path difference; the supplied distances already give that difference.

**Interpretation:** Equal arriving amplitudes would add to twice either amplitude. Geometric spreading can make the two amplitudes unequal, yet they are still phase-aligned. **TR:** İki hoparlör arasındaki uzaklığı değil, dinleyiciye giden iki yolun farkını kullan.

In [ ]:
#@title Optional numerical check — the algebra is explained above
# Worked Example 2: Two-source interference
f = 680    # Hz
v_sound = 340  # m/s
r1 = 4.0   # m
r2 = 4.5   # m

# Wavelength
lam = v_sound / f
print(f"Wavelength: lambda = v/f = {v_sound}/{f} = {lam:.3f} m")

# Path difference
delta_r = abs(r2 - r1)
print(f"Path difference: |r2 - r1| = |{r2} - {r1}| = {delta_r:.3f} m")

# How many wavelengths?
ratio = delta_r / lam
print(f"delta_r / lambda = {delta_r:.3f} / {lam:.3f} = {ratio:.2f}")

# Check
if abs(ratio - round(ratio)) < 0.01:
    print(f"\n=> delta_r = {round(ratio)}*lambda -> CONSTRUCTIVE interference (maximum)")
elif abs(ratio - round(ratio) - 0.5) < 0.01 or abs(ratio - round(ratio) + 0.5) < 0.01:
    m_val = int(ratio - 0.5) if ratio > 0 else int(ratio + 0.5)
    print(f"\n=> delta_r = ({m_val} + 1/2)*lambda -> DESTRUCTIVE interference (minimum)")
else:
    print(f"\n=> Partial interference (neither purely constructive nor destructive)")
    print(f"   Closest integer: {round(ratio)}, difference: {abs(ratio - round(ratio)):.2f}")

### 🎬 Engineering application: Vibration and Acoustics

#### Resonance in Structures

Understanding standing waves is critical in engineering:
- **Bridges** can respond strongly to loads near their natural frequencies; wind-driven instability is a separate mechanism
- **Machine shafts** have critical speeds where vibration amplitudes become dangerous
- **Building floors** are designed to avoid resonance with foot traffic

Below: explore how a bar fixed at both ends vibrates at its natural frequencies.

<a id="x13-problems"></a>

## 3. Problem set with step-by-step answers / Problem seti

Try the diagram and the symbolic equation before opening **Answer and steps**.

### Core problems (L1) / Temel problemler

Everyone should complete these; they follow the worked examples directly.

#### P1  ·  L1
A transverse wave on a string is described by $y(x,t) = 0.05\sin(3.0x - 12t)$ where $x$ and $y$ are in meters and $t$ is in seconds. Find the amplitude, wavelength, frequency, and wave speed.

**Türkçe — problem:** İpteki enine dalga $y(x,t)=0.05\sin(3.0x-12t)$ ile verilir; $x,y$ metre, $t$ saniye cinsindendir. Genliği, dalga boyunu, frekansı ve dalga hızını bulun.

<details><summary>Answer</summary>

$A = 0.05\,\mathrm{m}$; $\lambda = 2.094\,\mathrm{m}$; $f = 1.91\,\mathrm{Hz}$; $v = 4.0\,\mathrm{m/s}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P1 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

#### P2  ·  L1
A guitar string of length $0.65\,\mathrm{m}$ and mass $3.2\,\mathrm{g}$ is under $73\,\mathrm{N}$ of tension. What is the speed of transverse waves on this string, and what is the fundamental frequency?

**Türkçe — problem:** Uzunluğu $0.65\,\mathrm m$, kütlesi $3.2\,\mathrm g$ olan gitar teli $73\,\mathrm N$ gerilme altındadır. Enine dalga hızını ve temel frekansı bulun.

<details><summary>Answer</summary>

$$\mu = 3.2\times10^{-3}/0.65 = 4.9231\times10^{-3}\,\mathrm{kg/m},$$

so

$$v = \sqrt{73/\mu} = 121.8\,\mathrm{m/s}$$

and

$$f_1 = \frac{v}{2L} = 93.7\,\mathrm{Hz}.$$

**[CORRECTED]** previously 122.6 m/s and 94.3 Hz (about 0.7% high).


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P2 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

#### P3  ·  L1
Two identical speakers emit sound at $f = 850\,\mathrm{Hz}$ in phase. A listener is $5.0\,\mathrm{m}$ from one speaker and $5.5\,\mathrm{m}$ from the other. The speed of sound is $343\,\mathrm{m/s}$. Determine whether the interference is constructive, destructive, or partial.

**Türkçe — problem:** İki özdeş hoparlör $f=850\,\mathrm{Hz}$’de aynı fazda ses üretir. Dinleyici birine $5.0\,\mathrm m$, diğerine $5.5\,\mathrm m$ uzaktadır. Ses hızı $343\,\mathrm{m/s}$ ise girişimin yapıcı, yıkıcı veya kısmi olduğunu belirleyin.

<details><summary>Answer</summary>

Path difference $= 0.500\,\mathrm{m}$ $= 1.24\lambda$; partial interference (neither perfectly constructive nor destructive)


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P3 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

#### P4  ·  L1
A sound source produces an intensity of $I = 2.5 \times 10^{-5}\,\mathrm{W/m^2}$. Calculate the sound level in decibels. If the intensity doubles, what is the new decibel level?

**Türkçe — problem:** Bir ses kaynağının yoğunluğu $I=2.5\times10^{-5}\,\mathrm{W/m^2}$’dir. Ses düzeyini desibel cinsinden bulun. Yoğunluk iki katına çıkarsa yeni düzey kaç olur?

<details><summary>Answer</summary>

$\beta = 74.0\,\mathrm{dB}$; doubled: $\beta = 77.0\,\mathrm{dB}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P4 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

### Pause and explain / Dur ve açıkla

Before the intermediate problems, explain one core result to a partner.

#### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** An echo returns $0.010\,\mathrm s$ after emission. Sound speed is $340\,\mathrm{m/s}$; the target is stationary.

**Think — 1 minute:** Explain why multiplying speed by the recorded time is not yet the target distance.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

The sound travels to the target and back:
$$2d=vt\Rightarrow d=\frac{vt}{2}=\frac{340(0.010)}2=1.70\,\mathrm m.$$
The total travelled path is $3.40\,\mathrm m$, twice the one-way distance. A timestamp-based distance display and the physical reflection model must share this interpretation.

**Türkçe:** Süre gidiş ve dönüşün toplamıdır. Önce yolun $2d$ olduğunu yazdık, sonra ikiye böldük. Yazılım doğru çarpma yapsa bile yanlış fiziksel yol tanımı iki kat mesafe gösterebilir.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### Intermediate problems (L2) / Orta düzey

Combine two ideas from this week.

#### P5  ·  L2
A steel wire (density $7800\,\mathrm{kg/m^3}$, diameter $0.80\,\mathrm{mm}$) is stretched between two supports $1.50\,\mathrm{m}$ apart. The wire vibrates in its third harmonic at $440\,\mathrm{Hz}$ (concert A). Find (a) the wave speed, (b) the tension in the wire, and (c) the frequencies of the first five harmonics.

**Türkçe — problem:** Yoğunluğu $7800\,\mathrm{kg/m^3}$ ve çapı $0.80\,\mathrm{mm}$ olan çelik tel, araları $1.50\,\mathrm m$ olan destekler arasında gerilidir. Üçüncü harmonik frekansı $440\,\mathrm{Hz}$’dir. (a) Dalga hızını, (b) tel gerilmesini ve (c) ilk beş harmonik frekansını bulun.

<details><summary>Answer</summary>

**(a)** $v = 440\,\mathrm{m/s}$; **(b)** $T = 760\,\mathrm{N}$; **(c)** $f_1 = 146.7\,\mathrm{Hz}$, $f_2 = 293.3\,\mathrm{Hz}$, $f_3 = 440\,\mathrm{Hz}$, $f_4 = 586.7\,\mathrm{Hz}$, $f_5 = 733.3\,\mathrm{Hz}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P5 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

#### P6  ·  L2
An organ pipe open at both ends has a fundamental frequency of $256\,\mathrm{Hz}$ when the speed of sound is $343\,\mathrm{m/s}$. (a) What is the length of the pipe? (b) What is the fundamental frequency of a pipe of the same length that is closed at one end? (c) List the first three resonant frequencies for each pipe.

**Türkçe — problem:** İki ucu açık org borusunun temel frekansı, ses hızı $343\,\mathrm{m/s}$ iken $256\,\mathrm{Hz}$’dir. (a) Boru uzunluğunu, (b) aynı uzunlukta fakat bir ucu kapalı borunun temel frekansını bulun. (c) Her iki borunun ilk üç rezonans frekansını listeleyin.

<details><summary>Answer</summary>

**(a)** $L = 0.670\,\mathrm{m}$; **(b)**

$$f_1^\text{closed} = 128\,\mathrm{Hz};$$

**(c)** Open: $256, 512, 768\,\mathrm{Hz}$; Closed: $128, 384, 640\,\mathrm{Hz}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P6 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

#### P7  ·  L2
Two coherent sound sources ($f = 1200\,\mathrm{Hz}$, speed of sound $343\,\mathrm{m/s}$) are separated by $d = 1.5\,\mathrm{m}$. Find the angles (measured from the perpendicular bisector) at which the first two maxima and the first minimum occur in the far field.

**Türkçe — problem:** $f=1200\,\mathrm{Hz}$ frekanslı iki eşevreli ses kaynağı arasında $d=1.5\,\mathrm m$ vardır; ses hızı $343\,\mathrm{m/s}$’dir. Uzak alanda ilk iki maksimumun ve ilk minimumun açılarını dik orta doğrudan ölçerek bulun. Kullanılan başlangıç fazını ve merkezî maksimumu nasıl numaralandırdığınızı belirtin.

<details><summary>Answer</summary>

$$\lambda = 343/1200 = 0.2858\,\mathrm{m},$$

$d = 1.5\,\mathrm{m}$. Maxima at $d\sin\theta = n\lambda$, minima at $d\sin\theta = (n+\tfrac12)\lambda$: 0th max at $\theta = 0^\circ$; **1st minimum at $\theta = 5.47^\circ$**; **1st maximum at $\theta = 10.98^\circ$**; 2nd maximum at $\theta = 22.40^\circ$. **[CORRECTED]** previously '1st min 11.0$^\circ$, 1st max 22.2$^\circ$' — those are actually the 1st and 2nd *maxima*; the orders were shifted, not merely rounded.


</details>

**Interpretation note:** The usual listed angles assume the sources are in phase. Coherence alone only fixes a phase difference. Include the central maximum at $0^\circ$ and label the first two off-axis orders explicitly; matching negative angles occur on the other side.

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P7 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

#### P8  ·  L2
A wave pulse $y_1(x,t) = \frac{0.10}{1 + (x - 3t)^2}$ travels to the right, and $y_2(x,t) = \frac{-0.10}{1 + (x + 3t)^2}$ travels to the left (units: m, s). Find (a) the speed of each pulse, (b) the position and time where the pulses completely cancel, and (c) the displacement at $x = 0$ at $t = 0$.

**Türkçe — problem:** $y_1(x,t)=\dfrac{0.10}{1+(x-3t)^2}$ darbesi sağa, $y_2(x,t)=\dfrac{-0.10}{1+(x+3t)^2}$ darbesi sola ilerler; birimler metre ve saniyedir. (a) Her darbenin süratini, (b) darbelerin tamamen birbirini götürdüğü konum ve zamanı, (c) $x=0$, $t=0$ için toplam yer değiştirmeyi bulun.

<details><summary>Answer</summary>

**(a)** The pulse centres satisfy $x-3t=0$ and $x+3t=0$. Their velocities are therefore $+3.0\,\mathrm{m/s}$ and $-3.0\,\mathrm{m/s}$; both speeds are $3.0\,\mathrm{m/s}$.

**(b)** Their equal and opposite numerators cancel wherever their denominators agree:

$$\begin{aligned}
(x-3t)^2&=(x+3t)^2\\
x^2-6xt+9t^2&=x^2+6xt+9t^2\\
-12xt&=0\\
xt&=0.
\end{aligned}$$

Thus at $t=0$ the entire profiles cancel for every $x$, and their centres meet at $x=0$. Also $y_{\mathrm{net}}(0,t)=0$ for every time: $x=0$ is a location of continuing cancellation.

**(c)** $y(0,0)=0\,\mathrm m$. Zero displacement alone does not imply zero energy.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P8 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

### Challenge problems (L3) / İleri düzey

Engineering-style problems with several steps; useful preparation for the exams.

#### P9  ·  L3
A vibrating machine is mounted on a concrete floor (speed of longitudinal waves in concrete: $v_c = 3400\,\mathrm{m/s}$). The machine operates at $60\,\mathrm{Hz}$. (a) What is the wavelength of vibrations in the floor? (b) Sensitive equipment is located $25\,\mathrm{m}$ away. Assuming the wave intensity decreases as $I \propto 1/r^2$, by how many decibels is the vibration attenuated at that distance compared to $1.0\,\mathrm{m}$ from the machine? (c) If an isolation trench (air gap) of width $0.30\,\mathrm{m}$ is cut into the floor between the machine and the equipment, estimate whether it is effective. Compare the trench width to the wavelength.

**Türkçe — problem:** $60\,\mathrm{Hz}$’de çalışan makine, boyuna dalga hızı $v_c=3400\,\mathrm{m/s}$ olan beton zemindedir. (a) Zemindeki dalga boyunu bulun. (b) $I\propto1/r^2$ kabul ederek $25\,\mathrm m$ uzaklıktaki ekipmana ulaşan titreşimin, $1.0\,\mathrm m$ uzaklığa göre kaç dB zayıfladığını bulun. (c) Araya açılan $0.30\,\mathrm m$ genişlikte hava boşluğu hendeğinin etkinliğini, genişliği dalga boyuyla karşılaştırarak değerlendirin; hangi ek bilgilere ihtiyaç olduğunu belirtin.

<details><summary>Answer</summary>

**(a)** $\lambda=56.6667\,\mathrm{m}$. **(b)** Signed level change

$$\Delta\beta=10\log_{10}(1/625)=-27.9588\,\mathrm{dB},$$

or 27.96 dB attenuation. **(c)** $w/\lambda=0.005294\ll1$. Width alone does not establish useful isolation; trench depth, extent, wave type, impedance and remaining mechanical paths are needed. A definite effectiveness claim cannot be obtained from these data.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P9 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

#### P10  ·  L3
A string of length $L = 2.0\,\mathrm{m}$, linear density $\mu = 0.010\,\mathrm{kg/m}$, and tension $T = 40\,\mathrm{N}$ is plucked so that its initial shape is a triangle with peak displacement $h = 0.02\,\mathrm{m}$ at $x = L/3$. This shape can be decomposed into Fourier harmonics. (a) Find the fundamental frequency and wave speed. (b) The Fourier coefficient of the $n$-th harmonic for this triangular pluck is $b_n = \frac{9h}{n^2\pi^2}\sin\!\left(\frac{n\pi}{3}\right)$. Calculate $b_1$, $b_2$, $b_3$. (c) Which harmonics are absent and why?

**Türkçe — problem:** $L=2.0\,\mathrm m$, $\mu=0.010\,\mathrm{kg/m}$ ve $T=40\,\mathrm N$ olan tel, $x=L/3$ konumunda $h=0.02\,\mathrm m$ tepeye sahip üçgen biçimde çekilip bırakılır. (a) Temel frekansı ve dalga hızını bulun. (b) Verilen $b_n=\frac{9h}{n^2\pi^2}\sin(\frac{n\pi}{3})$ Fourier katsayısıyla $b_1,b_2,b_3$’ü hesaplayın. (c) Hangi harmoniklerin bulunmadığını ve nedenini açıklayın.

<details><summary>Answer</summary>

**(a)**

$$v = \sqrt{\frac{T}{\mu}} = 63.2\,\mathrm{m/s},$$

$$f_1 = \frac{v}{2L} = 15.8\,\mathrm{Hz}.$$

**(b)** Using the corrected coefficient

$$b_n = \dfrac{9h}{n^2\pi^2}\sin\!\left(\dfrac{n\pi}{3}\right)$$

: $b_1 = 15.794\,\mathrm{mm}$, $b_2 = 3.949\,\mathrm{mm}$, $b_3 = 0$. **(c)** $b_3 = b_6 = b_9 = \cdots = 0$ (all multiples of 3), because the pluck point $x_p = L/3$ is a node of those harmonics. **[CORRECTED]** the coefficient formula originally printed in this problem carried an extra factor of 2 in the denominator, giving values half the correct size. The projection integral

$$b_n = \frac{2}{L}\int_0^L y(x,0)\sin\frac{n\pi x}{L}dx$$

gives the form above; summing it reconstructs the true 20 mm pluck (the halved version reconstructs 10 mm).


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 13 P10 in the solutions collection (file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026).

## Solutions / Çözümler

Complete worked solutions: Module 13 (Waves and sound), file `Week_13_Python_Solutions.ipynb`, opens 18 December 2026 on the course page.

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)